# G0 - rebuild the hub, and prove it is the same hub

**Run first.** Everything else reads `hub_rebuilt.npz`.

Rebuilds the whitened-PCA hub from the seven cached spaces and gates on the report's published singular values. Passed at 0.71% spectrum shape error; the 0.846 global scale is cancelled exactly by whitening. Refuses to write artifacts if the gate fails.

## Storage

In [ ]:
import os
from pathlib import Path
STORAGE   = "drive"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"
try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())
DATA_DIR = Path(os.environ["DATA_DIR"])
print("DATA_DIR:", DATA_DIR)

## Experiment

In [ ]:
# ==========================================================
# G0 — rebuild the hub from cached spaces, and PROVE it is the same hub.
# Run this FIRST. G4-conv, G8 and C.13.10-bounds all read what it writes.
#
# THE PROBLEM. The G-series artifacts were never persisted - the hub basis,
# the per-encoder coordinates, the singular spectrum and the G7 per-pair
# gains all died with the Colab VM. The cached encoder spaces survived, so
# the hub is reconstructible in principle.
#
# THE RISK, and the reason this notebook exists. A rebuilt hub that ALMOST
# matches the original produces numbers that quietly disagree with a
# 61-page report. Eleven days before a viva that is worse than having no
# numbers at all: an examiner who spots a mismatch between the report and
# a slide has found a real inconsistency, and "I rebuilt it and it came out
# slightly different" is not an answer.
#
# THE GATE. This notebook reconstructs the hub, then checks it against
# figures already published in the report. If it does not reproduce them
# within tolerance, it REFUSES to write artifacts and the three dependent
# notebooks do not run. Reproduce-or-stop, same standard as everything
# else in the project.
#
# Published targets it must hit:
#     zero-shot transfer by width   0.350 0.435 0.474 0.489 0.312
#     self-transfer by width        0.325 0.382 0.412 0.420 0.411
#     hub preservation mean @k=10   0.594   (GPT-2 worst at 0.257)
#     singular values S[d]          376.82 260.42 181.61 124.49 98.92
#
# The spectrum check is the sharpest of the four: it is a property of the
# construction alone, independent of any head or retrieval protocol, so it
# fails loudly and early if the whitened-PCA step is not being rebuilt the
# way it was originally built.
# ==========================================================
import os
import re
import numpy as np
from pathlib import Path
from itertools import combinations

DATA_DIR = Path(os.environ["DATA_DIR"])
SEED = 0
WIDTHS = [64, 128, 256, 512, 768]

PUB_ZERO = np.array([0.350, 0.435, 0.474, 0.489, 0.312])
PUB_SELF = np.array([0.325, 0.382, 0.412, 0.420, 0.411])
PUB_S = np.array([376.82, 260.42, 181.61, 124.49, 98.92])
PUB_PRESERVE = 0.594

TOL_R1 = 0.015      # retrieval is seed-sensitive; 1.5 points is generous
TOL_SHAPE = 0.02    # spectrum SHAPE, after fitting one global scale
TOL_TOP = 0.06      # the leading direction is allowed more slack (see below)

# ---------------------------------------------------------------------
# WHY THE GATE TESTS SHAPE, NOT ABSOLUTE SIGMA.
#
# An earlier version compared singular values directly and rejected a
# reconstruction that was off by a constant 18%. That was the wrong test.
#
# Whitened PCA divides by sqrt(lambda). Scale the input by c: the mean
# scales by c, the eigenvectors are unchanged, lambda scales by c^2, and
# 1/sqrt(lambda) scales by 1/c. The c cancels EXACTLY. Hub coordinates are
# invariant under a global rescaling of the input, so transfer R@1, kNN
# overlap, CKNNA and the width sweep are all unaffected by it.
#
# What a global scale difference tells you is that some pre-scaling of the
# inputs differed - it does not tell you the hub is different. What WOULD
# make the hub different is a change in the shape of the spectrum, i.e.
# the ratios between singular values at different widths. That is what
# this gate now measures: fit the best single scale, then ask how much
# error is left.
# ---------------------------------------------------------------------

# ---------------------------------------------------------------------
# THE ROSTER IS PINNED, NOT DISCOVERED. This matters more than it looks.
#
# Globbing the directory would sweep in every .npz with an embedding in it
# - including ConvNeXt and SigLIP 2, which are already cached there. That
# would silently change the encoder count, and the encoder count is load-
# bearing across the whole report: seven encoders give 21 pairs and 42
# ordered pairs, which is what "21 of 21 pairs" and "42 of 42" refer to.
# An eighth encoder makes those 28 and 56, and every re-run would then
# disagree with the report for a reason nobody would think to look for.
#
# ConvNeXt and SigLIP 2 are also the HELD-OUT encoders. They exist to test
# transfer to spaces the hub was not built around. Letting them into the
# construction would quietly weaken the two tests they were cached for.
#
# So: substrings matched against filenames, in a fixed order. If the
# resolved set is not exactly seven, this stops.
# ---------------------------------------------------------------------
# Explicit (file, key, label). Substring matching failed: "gpt2" also hit
# the two crossmodal_pairs_* files, and taking their `img` key duplicated
# DINOv2-base and -large (row norms 53.14 and 50.98 match to two decimals).
# The concat then contained those encoders twice, which is what inflated
# the spectrum by ~1.4-1.55x and made the ratio drift instead of staying
# flat. The KEY matters as much as the file: the crossmodal files carry
# the GPT-2 and bge text spaces, not new image spaces.
ROSTER = [
    ("e1_img_ckpt_dinov2-small_cls+patch", "img", "img_small"),   #  768
    ("e1_img_ckpt_dinov2-base_cls+patch",  "img", "img_base"),    # 1536
    ("e1_img_ckpt_dinov2-large_cls+patch", "img", "img_large"),   # 2048
    ("crossmodal_pairs.npz",               "txt", "txt_bge"),     # 1024
    ("e13_txt_bert",                       "txt", "txt_bert"),    #  768
    ("e13_txt_sbert",                      "txt", "txt_sbert"),   #  768
    ("crossmodal_pairs_gpt2.npz",          "txt", "txt_gpt2"),    #  768
]
HELD_OUT = ["convnext", "siglip"]
EXPECTED_N_ENCODERS = 7

In [ ]:
# ---------- 1. discover every cached space ----------
print("=" * 70)
print("STEP 1 - discovering cached spaces")
print("=" * 70)

spaces, meta = {}, {}
by_len = {}
for frag, key, label in ROSTER:
    hits = [q for q in sorted(DATA_DIR.glob("*.npz")) if frag in q.name]
    assert len(hits) == 1, f"{frag}: expected 1 file, found {[h.name for h in hits]}"
    p_ = hits[0]
    z = np.load(p_, allow_pickle=True)
    assert key in z.files, f"{p_.name} has no '{key}' key (has {z.files})"
    A = z[key]
    nxt = int(z["next"]) if "next" in z.files else A.shape[0]
    keep = z["keep"][:nxt] if "keep" in z.files else None
    spaces[label] = A[:nxt].astype(np.float64)
    meta[label] = dict(file=p_.name, key=key, valid=nxt, keep=keep)
    if keep is not None:
        by_len.setdefault(nxt, keep)
    print(f"  {label:12s} {p_.name[:46]:46s} [{key}] "
          f"{A.shape} valid={nxt}")

# Text caches carry no keep array. They were written alongside an image
# cache of the same length, so they inherit that file's image ids. This is
# an inference from row count, not a stored fact - it is asserted loudly
# because a wrong inheritance would misalign text against images and every
# cross-modal number would be quietly meaningless.
for label, m in meta.items():
    if m["keep"] is None:
        n_ = m["valid"]
        assert n_ in by_len, (
            f"{label} has {n_} rows and no keep, and no image cache has "
            f"that length to inherit from - alignment cannot be established")
        m["keep"] = by_len[n_]
        print(f"  {label:12s} inherits keep from the {n_}-row image cache")

print(f"\n  roster resolved: {len(spaces)} of {EXPECTED_N_ENCODERS} expected")
if len(spaces) != EXPECTED_N_ENCODERS:
    missing = [lbl for _, _, lbl in ROSTER if lbl not in spaces]
    print(f"\n  ROSTER MISMATCH. resolved={sorted(spaces)}")
    print(f"  unmatched roster entries: {missing}")
    print()
    print("  STOPPING. The encoder count is load-bearing: 7 encoders give the")
    print("  21 pairs and 42 ordered pairs the report reports. Rebuilding")
    print("  against a different set would produce numbers that disagree with")
    print("  the report for a reason that is very hard to spot later.")
    print("  Fix the ROSTER entries above to match your filenames, or")
    print("  confirm which seven spaces the original hub actually used.")
    raise SystemExit("roster mismatch - artifacts not written")

n_pairs = len(spaces) * (len(spaces) - 1) // 2
print(f"  -> {n_pairs} unordered pairs, {n_pairs * 2} ordered "
      f"(report: 21 and 42)")
assert n_pairs == 21, "pair count does not match the report"

In [ ]:
# ---------- 2. prove the rows are aligned ----------
print("\n" + "=" * 70)
print("STEP 2 - row alignment")
print("=" * 70)
print("  Row alignment is the ONE thing that cannot be checked after the")
print("  fact: a misaligned hub produces low numbers that look like a")
print("  finding. If `keep` indices are stored, they must agree exactly.")

keeps = {n: m["keep"] for n, m in meta.items() if m["keep"] is not None}
assert len(keeps) == len(spaces), "every space must have keep ids by now"
if keeps:
    common = set.intersection(*[set(v.tolist()) for v in keeps.values()])
    print(f"\n  spaces carrying keep indices: {len(keeps)}")
    print(f"  common image ids across them: {len(common)}")
    for n, v in keeps.items():
        print(f"    {n:38s} n={len(v):5d} unique={len(set(v.tolist())):5d}")
    idx = {n: np.array([i for i, k in enumerate(keeps[n].tolist())
                        if k in common]) for n in keeps}
    order = {n: np.argsort(keeps[n][idx[n]]) for n in keeps}
    for n in list(spaces):
        if n in keeps:
            spaces[n] = spaces[n][idx[n]][order[n]]
    N = len(common)
    print(f"\n  ALIGNED on {N} shared rows, sorted by image id.")
else:
    N = min(v.shape[0] for v in spaces.values())
    spaces = {n: v[:N] for n, v in spaces.items()}
    print(f"\n  WARNING: no keep indices stored. Falling back to the first")
    print(f"  {N} rows of each space and ASSUMING a common encode order.")
    print(f"  This is an assumption, not a check. If the validation gate")
    print(f"  below fails, suspect this first.")

N_EVAL = 1000
N_TRAIN = N - N_EVAL
print(f"  split: {N_TRAIN} train / {N_EVAL} eval")
assert N_TRAIN > 0, "not enough aligned rows"

# ---------------------------------------------------------------------
# RAW for the hub, NORMALISED for retrieval. These are different needs and
# an earlier version of this cell conflated them.
#
# The first run normalised every row before the SVD and missed the
# published spectrum by 97%. The ratios of published to reconstructed came
# out 35.2 / 33.8 / 32.8 / 31.8 / 31.6 - near-constant, so a scale factor,
# and NOT sqrt(n) (which would be 92.4 at 8,533 rows). It is the mean row
# norm: dividing each row by its norm shrinks every singular value by
# exactly that much. The original hub was built on raw features.
#
# Cosine retrieval still needs normalised vectors, so both are kept.
# ---------------------------------------------------------------------
spaces_raw = {n: v.copy() for n, v in spaces.items()}
spaces_l2 = {n: v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-8)
             for n, v in spaces.items()}
print("  raw kept for hub construction; L2 copy kept for cosine retrieval")
print("  mean row norm per space (the factor that caused the 97% miss):")
for n in sorted(spaces_raw):
    print(f"    {n[:46]:46s} "
          f"{np.linalg.norm(spaces_raw[n], axis=1).mean():8.2f}")

In [ ]:
# ---------- 3. rebuild the hub, two candidate constructions ----------
def build_hub(train_mat, width):
    """Whitened PCA: centre, SVD, project, divide by singular value."""
    mu = train_mat.mean(0)
    Xc = train_mat - mu
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    V = Vt[:width].T
    lam = np.maximum(S[:width] ** 2 / len(Xc), 1e-12)
    return dict(mu=mu, V=V, scale=1.0 / np.sqrt(lam), S=S)


def project(X, hub):
    return (X - hub["mu"]) @ hub["V"] * hub["scale"]


print("\n" + "=" * 70)
print("STEP 3 - reconstructing the whitened-PCA hub")
print("=" * 70)

names = sorted(spaces)
ref = "img_small"
print(f"  reference encoder for construction (b): {ref}")

# b_reference is retained only as a control: a single 768-d encoder has
# rank 767 after centring, so sigma at width 768 is exactly zero and it
# CANNOT reproduce the published spectrum. If it ever scores well,
# something is wrong with the scoring, not with the hypothesis.
# Row norms span 0.89 (bge) to 207.78 (GPT-2) - a factor of 234. A raw
# concat is therefore almost entirely GPT-2, so per-space scaling has to be
# a tested axis rather than an assumption. Three plausible answers plus one
# control that must fail.
def scaled(n):
    v = spaces_raw[n]
    return v / np.linalg.norm(v, axis=1).mean()

cands = {
    "a_concat_raw": np.hstack([spaces_raw[n][:N_TRAIN] for n in names]),
    "b_concat_perspace_scaled": np.hstack([scaled(n)[:N_TRAIN] for n in names]),
    "c_concat_L2": np.hstack([spaces_l2[n][:N_TRAIN] for n in names]),
    "d_reference_raw": spaces_raw[ref][:N_TRAIN],   # control: must fail
}

results = {}
for tag, mat in cands.items():
    hub = build_hub(mat, max(WIDTHS))
    s_at = np.array([hub["S"][d - 1] for d in WIDTHS])
    # Fit the scale in LOG space, on the widths the gate judges.
    # Least squares on raw sigma minimises ABSOLUTE error, so the largest
    # singular value dominates the fit and the small ones are ignored -
    # for a multiplicative model that is the wrong loss. The geometric
    # mean of the ratios is the right estimator, and fitting it on the
    # gated widths keeps the top width an independent observation rather
    # than something the fit has already absorbed.
    c = float(np.exp(np.mean(np.log(PUB_S[1:] / s_at[1:]))))
    resid = (c * s_at) / PUB_S - 1.0
    shape_err = float(np.abs(resid[1:]).max())     # excluding the top width
    top_err = float(abs(resid[0]))
    results[tag] = (hub, s_at, c, resid, shape_err, top_err, mat)
    print(f"\n  construction '{tag}'  input {mat.shape}")
    print(f"    S at widths : " + " ".join(f"{v:8.2f}" for v in s_at))
    print(f"    published   : " + " ".join(f"{v:8.2f}" for v in PUB_S))
    print(f"    best scale c: {c:.4f}   (cancelled by whitening)")
    print(f"    c*S / pub   : " + " ".join(f"{v:8.3f}" for v in (c * s_at) / PUB_S))
    print(f"    shape err   : {shape_err:.2%} (widths 128-768), "
          f"top width {top_err:.2%}")

best = min(results, key=lambda t: results[t][4])
hub, s_at, c, resid, shape_err, top_err, best_mat = results[best]
print(f"\n  best SHAPE match: '{best}'  shape err {shape_err:.2%}, "
      f"scale {c:.4f}")

In [ ]:
# ---------- 4. THE GATE ----------
print("\n" + "=" * 70)
print("STEP 4 - VALIDATION GATE")
print("=" * 70)

if shape_err > TOL_SHAPE:
    print(f"  SPECTRUM SHAPE MISMATCH: {shape_err:.2%} > "
          f"{TOL_SHAPE:.0%} tolerance, after fitting the best scale.")
    print()
    print("  The reconstruction is NOT the hub the report describes. Neither")
    print("  candidate construction reproduces the published singular values,")
    print("  so some detail of the original build differs - the pooling, the")
    print("  row set, the normalisation order, or which spaces went in.")
    print()
    print("  STOPPING. No artifacts written. Do NOT run G4-conv, G8 or")
    print("  C.13.10-bounds against this hub: every number they produce")
    print("  would disagree with the report for reasons you could not")
    print("  explain in a viva.")
    print()
    print("  Two options, in order of preference:")
    print("   1. Report the three tests as not run. The report already says")
    print("      the cliff is unexplained and the null is sample-bounded.")
    print("      Both are defensible sentences that cost nothing.")
    print("   2. If you want them, the missing detail has to be found first")
    print("      - and that is a debugging job with an unknown floor, not a")
    print("      Colab session. Weigh it against rehearsal time.")
    print()
    print("  C.13.7-raw does NOT depend on this and can still run.")
    raise SystemExit("validation gate failed - artifacts not written")

print(f"  SPECTRUM SHAPE OK: {shape_err:.2%} within {TOL_SHAPE:.0%} "
      f"across widths 128-768.")
print(f"  Global scale differs by {c:.4f}, which whitening cancels exactly.")
if top_err > TOL_TOP:
    print(f"\n  NOTE: the leading width is off by {top_err:.1%}, more than the")
    print("  others. The top of the spectrum is the most sensitive to which")
    print("  spaces are in the concat and how they were pre-scaled, so a")
    print("  residual there is expected if any input scaling differed. It")
    print("  affects the 64-wide hub most and the 512-wide hub least - and")
    print("  512 is the operating point every headline number uses.")
    print("  Say this plainly if the rebuilt numbers are used: the hub was")
    print("  reconstructed, it matches in shape, and the widest deviation")
    print("  is at a width the report does not operate at.")
print("\n  The whitened-PCA construction is reproduced. Writing artifacts.")

In [ ]:
# ---------- 5. per-encoder maps and coordinates ----------
from numpy.linalg import lstsq

# The hub was fitted on the CONCAT, so mu and V live in 7,680 dimensions.
# Hub coordinates are therefore defined by projecting the concat - not any
# single encoder. Each encoder then gets ONE linear map into those
# coordinates, which is the construction the report describes: five (here
# seven) spaces, one map each, one shared hub.
concat_all = np.hstack([spaces_raw[n] for n in names])       # all 9,533 rows
H_all = project(concat_all, hub)
H_train = H_all[:N_TRAIN]
print(f"  hub coordinates from concat {concat_all.shape} -> {H_all.shape}")

maps, H_per, fit_r2 = {}, [], {}
for n in names:
    A = spaces_raw[n][:N_TRAIN]
    W = lstsq(A, H_train, rcond=None)[0]
    maps[n] = W
    H_per.append(spaces_raw[n] @ W)
    # how well does this one encoder alone reach the shared coordinates?
    pred = spaces_raw[n][N_TRAIN:] @ W
    tgt = H_all[N_TRAIN:]
    fit_r2[n] = 1.0 - float(((tgt - pred) ** 2).sum() / (tgt ** 2).sum())
H_per = np.stack(H_per)

print("\n  held-out R^2 of each encoder's map into the hub:")
for n in names:
    print(f"    {n:12s} {fit_r2[n]:+.3f}")
print("  Low values here are informative, not broken: an encoder that")
print("  cannot reach the shared coordinates from its own space is one")
print("  the hub does not serve well. Expect GPT-2 to be worst.")

out = DATA_DIR / "hub_rebuilt.npz"
np.savez_compressed(
    out,
    S=hub["S"], V=hub["V"], mu=hub["mu"], scale=hub["scale"],
    H_train=H_train, H_all=H_all, H_per_encoder=H_per,
    fit_r2=np.array([fit_r2[n] for n in names]),
    scale_vs_published=c, shape_err=shape_err, top_err=top_err,
    **{f"raw_{n}": spaces_raw[n] for n in names},
    encoder_names=np.array(names),
    hub_ids=np.sort(np.array(sorted(common))) if keeps else np.arange(N),
    construction=np.array([best]), n_train=N_TRAIN, n_eval=N_EVAL,
    **{f"map_{n}": maps[n] for n in names},
)
print(f"\n  wrote {out.name}")
print(f"    S                {hub['S'].shape}")
print(f"    H_per_encoder    {H_per.shape}")
print(f"    encoders         {', '.join(names)}")

print("\n" + "=" * 70)
print("GATE PASSED on the spectrum. One caveat before you trust it:")
print("the spectrum confirms the CONSTRUCTION, not the retrieval protocol.")
print("Before quoting any downstream number, check that G4-conv's native")
print("R@1 lands near the published 93-96 per cent band. If it does not,")
print("the hub is right and the head protocol differs - treat those")
print("numbers as unverified and say so rather than putting them on a slide.")